In [1]:
import pandas as pd
from difflib import SequenceMatcher
from fuzzywuzzy import fuzz
import numpy as np
import re
import textdistance
from sklearn.cluster import AgglomerativeClustering  

c:\Users\Matthew.Vaughn\AppData\Local\Programs\Python\Python311\Lib\site-packages\fuzzywuzzy\fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [57]:
#read excel file into pandas
df = pd.read_excel(r"C:\Users\Matthew.Vaughn\OneDrive - Berkeley Research Group, LLC\Business Development\Daubert Motions\Version 3 (2024.02.06)\Daubert Motions by Law Firm, Output.xlsx", sheet_name='Unique Firm Names')
df['Firm'] = df['Firm'].str.strip()
df['Firm'] = df['Firm'].replace('  ',' ',regex=True)
df['Firm'] = df['Firm'].replace('   ',' ',regex=True)
df['Firm'] = df['Firm'].replace('\+',' ',regex=True)
texts = df['Firm'].to_list()


In [77]:
def normalize(text):
  """ Keep only lower-cased text and numbers"""
  return text.lower()

def group_texts(texts, threshold=0.41): 
  """ Replace each text with the representative of its cluster"""
  normalized_texts = np.array([normalize(text) for text in texts])
  distances = 1 - np.array([
      [textdistance.jaro_winkler(one, another) for one in normalized_texts] 
      for another in normalized_texts
  ])
  clustering = AgglomerativeClustering(
    distance_threshold=threshold, metric ="euclidean",linkage="complete", n_clusters=None
  ).fit(distances)
  centers = dict()
  
  for cluster_id in set(clustering.labels_):
    index = clustering.labels_ == cluster_id
    centrality = distances[:, index][index].sum(axis=1)
    centers[cluster_id] = normalized_texts[index][centrality.argmin()]
  return [centers[i] for i in clustering.labels_]

standardized_text = (group_texts(texts))
standardized_text = [x.upper() for x in standardized_text]
standardized_text = list(set(standardized_text))
standardized_text



c:\Users\Matthew.Vaughn\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\cluster\_agglomerative.py:585: ClusterWarning: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  out = hierarchy.linkage(X, method=linkage, metric=affinity)


['SHAW KELLER',
 'DYKEMA UPSHAW',
 'SHORE CHAN BRAGALONE DEPUMPO',
 'ENDURANCE LAW GROUP',
 'SIDDIQUI',
 'CAPPELLO NOEL',
 'WARNER NORCROSS JUDD',
 'TRUMAN HAYMAKER FENTON',
 'DAVIDSON BERQUIST JACKSON GOWDEY',
 'WILMERHALE DORR',
 'FABIAN VANCOTT',
 'PHILLIPS GOLDMAN MCLAUGHLIN HALL',
 'DURIE DANGRI',
 'WELTCHEK MALLANHAN WELTCHEK',
 'BANOWSKY LEVINE',
 'VAN NEST PETERS',
 'ADLI',
 'LANGDON EMISON',
 'WILLIAMS LAW FIRM',
 'NIRO HALLER NIRO',
 'TYZ LAW GROUP',
 'ARSHT TUNNELL',
 'MCNELLY LAW GROUP',
 'FEINBERG DAY ALBERTI THOMPSON',
 'NELSON BUMGARDNER',
 'FENWICK WEST',
 'CARLSON CASPERS',
 'KREBS FARLEY DRY',
 'GOOSMANN',
 'SKIERMONT PUCKETT',
 'ICE MILLER',
 'HAUG PARTNERS',
 'JONAK | PUGH',
 'WILMER HALE',
 'BINGHAM MCCUTCHEN',
 'FINNEGAN',
 'BARLEY SNYDER',
 'RICHARDS LAYTON FINGER',
 'RICE SAITO',
 'ECHELON',
 'RASHAUNA NORMENT',
 'VENABLE',
 'BAHOU MILLER',
 'FOX ROTHSCHILD',
 'BRAGALONE CONROY',
 'KRUMHOLZ  MENTLIK',
 'COLLINS EDMONDS SCHLATHER',
 'CARVER DARDEN KORETZKY TESSIE

In [78]:
def find_and_standardize_misspellings(text):
    similarity_threshold = 85
    for x in standardized_text:
        if fuzz.ratio(x,text) > similarity_threshold:
            return x
        else:
            pass

df['Standardized_Firm'] = df['Firm'].apply(find_and_standardize_misspellings)

In [79]:
df

,Firm,Standardized_Firm
0,ADLER POLLOCK SHEEHAN,ADLER POLLOCK SHEEHAN
1,ADLI,ADLI
2,AGILITY IP LAW,AGILITY IP LAW
3,AHMAD ZAVITSANOS ANAIPAKOS ALAVI MENSING,AHMAD ZAVITSANOS ANAIPAKOS ALAVI MENSING
4,AKERMAN,AKERMAN
...,...,...
841,YOUNG BASILE HANLON MACFARLANE,YOUNG BASILE HANLON MACFARLANE
842,YOUNG BASILE HANLON MACFARLANE,YOUNG BASILE HANLON MACFARLANE
843,YOUNG CONAWAY STARGATT TAYLOR,YOUNG CONAWAY STARGATT TAYLOR
844,YULCHON,YULCHON


In [80]:
df.to_excel(r'C:\Users\Matthew.Vaughn\OneDrive - Berkeley Research Group, LLC\Business Development\Daubert Motions\Version 3 (2024.02.06)\Standardized Names.xlsx')